# Notebook 1 — FSM v6: Mapeo Semántico y Trayectorias

Genera `trajectories_full.csv`, `trajectories_compact.csv`, `trajectories_summary.csv`

In [ ]:
import os, numpy as np, random, pandas as pd
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
BASE_PATH = '.'
INPUTS  = os.path.join(BASE_PATH, 'data')
OUTPUTS = os.path.join(BASE_PATH, 'outputs')
os.makedirs(OUTPUTS, exist_ok=True)
ID_COL = 'email'; STATE_COL = 'state'; TIME_COL = 'timestamp'
print(f'INPUTS: {INPUTS} | OUTPUTS: {OUTPUTS} | SEED: {SEED}')


In [ ]:
logs = pd.read_csv(f"{INPUTS}/logs_moodle_anon.csv", on_bad_lines='skip')
logs = logs.rename(columns={
    'nombre evento': 'eventname', 'descripcion': 'description',
    'componente': 'component',   'contexto del evento': 'contexto'
})
logs['timestamp'] = pd.to_datetime(logs['hora'].astype(str).str.strip(),
                                    dayfirst=True, format='mixed', errors='coerce')
for col in ['eventname','description','contexto','component']:
    logs[col] = logs[col].astype(str).str.lower()
logs = logs.sort_values([ID_COL, 'timestamp'])
print(f'Logs cargados: {len(logs)} eventos, {logs[ID_COL].nunique()} alumnos')
logs.head()


## Mapeo Semántico FSM v6

In [ ]:
def map_state(row):
    desc = row['description']; ctx = row['contexto']; comp = row['component']
    if 'cuestionario' in comp:
        return 'EVAL' if 'submitted the attempt' in desc else 'QUIZ'
    if any(k in ctx for k in ['práctica','practica','consola','python',
                               'mlp','kmeans','rbf','som','lvq','aprendizaje','clasificador']):
        return 'PRACT'
    if any(k in ctx for k in ['video','tutorial','síntesis','sintesis','desarrollo']): return 'REC'
    if "viewed the 'resource'" in desc or "viewed the 'page'" in desc: return 'REC'
    if any(k in desc for k in ['updated the grade','graded']): return 'FB'
    if any(k in desc for k in ['viewed the section','course viewed']): return 'NAV'
    return 'OTHER'

logs[STATE_COL] = logs.apply(map_state, axis=1)
print('Distribución de estados:')
print(logs[STATE_COL].value_counts())
logs[[TIME_COL, ID_COL, 'contexto', 'eventname', STATE_COL]].head()


In [ ]:
# Trayectorias full
traj_full = logs[[ID_COL, 'timestamp', STATE_COL]].copy()
traj_full.to_csv(f"{OUTPUTS}/trajectories_full.csv", index=False)

# Trayectorias compactas (sin repeticiones consecutivas)
compact_rows = []
for email, g in traj_full.groupby(ID_COL):
    seq = g[STATE_COL].tolist()
    comp = [seq[0]]
    for s in seq[1:]:
        if s != comp[-1]: comp.append(s)
    for i, st in enumerate(comp):
        compact_rows.append([email, i, st])
compact_df = pd.DataFrame(compact_rows, columns=[ID_COL,'order',STATE_COL])
compact_df.to_csv(f"{OUTPUTS}/trajectories_compact.csv", index=False)

# Summary: proporciones por estado
summary = (traj_full.groupby(ID_COL)[STATE_COL]
           .value_counts(normalize=True).unstack().fillna(0).reset_index())
summary.to_csv(f"{OUTPUTS}/trajectories_summary.csv", index=False)

print(f'trajectories_full.csv      -> {len(traj_full)} filas')
print(f'trajectories_compact.csv   -> {len(compact_df)} filas')
print(f'trajectories_summary.csv   -> {summary.shape[0]} alumnos x {summary.shape[1]-1} estados')
print('FSM v6 generado correctamente.')
